# Grounding Service Search
This notebook demonstrates how to search the Grounding Service API, select datasets, and display results in a tabular format using standard Python and pandas.

#### First we need to install dependencies

In [ ]:
# Upgrade pip and install dependencies
%pip install --upgrade pip
%pip install aiohttp requests pandas ipykernel

# echo "Dependencies installed in the virtual environment."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.5 MB/s  0:00:00


In [ ]:
"""Notebook for the Adaptive Search API and displaying results.

This notebook provides a user interface for querying Adaptive Search,
selecting datasets, and viewing results in a tabular format.
"""

import asyncio
import aiohttp
import os
import time
import requests
import pandas as pd

ADAPTIVE_SEARCH_API_HOST = os.getenv(
    "ADAPTIVE_SEARCH_API_HOST", "https://grounding.kensho.com"
)
ADAPTIVE_SEARCH_API_HOST

'https://grounding.kensho.com'

We need `Access/Bearer Token` to call the `ADAPTIVE_SEARCH_API_HOST`.

One can get it on https://grounding.kensho.com/login/.
This token will expire in 1hr.

Alternatively, use the `Refresh Token`, which does not expire.

In [ ]:
import getpass

# Prompt user for refresh token if not already set in environment
if not os.getenv("ADAPTIVE_SEARCH_REFRESH_TOKEN"):
    refresh_token = getpass.getpass("Enter your ADAPTIVE_SEARCH_REFRESH_TOKEN. Enter to skip: ")
else:
    refresh_token = os.getenv("ADAPTIVE_SEARCH_REFRESH_TOKEN")
os.environ["ADAPTIVE_SEARCH_REFRESH_TOKEN"] = refresh_token

In [ ]:
def get_adaptive_search_access_token() -> str:
    """Get access token from Okta for Adaptive Search service."""
    if not refresh_token:
        return getpass.getpass("Enter your Access/Bearer Token: ")
    refresh_url = f"{ADAPTIVE_SEARCH_API_HOST}/oauth2/refresh"
    refresh_response = requests.post(
        refresh_url,
        json={"refresh_token": os.getenv('ADAPTIVE_SEARCH_REFRESH_TOKEN')}
    )
    refresh_response.raise_for_status()
    return refresh_response.json()
access_token = get_adaptive_search_access_token()
os.environ["ADAPTIVE_SEARCH_ACCESS_TOKEN"] = access_token

In [ ]:
def load_available_datasets():
    """Load available datasets from the remote agents API."""
    try:
        response = requests.get(
            f"{ADAPTIVE_SEARCH_API_HOST}/api/v1/agents",
            headers={
                "Authorization": f"Bearer {os.getenv('ADAPTIVE_SEARCH_ACCESS_TOKEN')}",
                "Content-Type": "application/json",
            },
            timeout=10,
        )
        response.raise_for_status()
        agents = response.json()
        return [
            {
                "name": agent["name"],
                "description": agent.get("short_description", ""),
                "example_queries": agent.get("example_queries", []),
            }
            for agent in agents["agents"]
        ]
    except Exception as e:
        print(f"Error loading datasets: {str(e)}")
        return []

AVAILABLE_DATASETS = load_available_datasets()

## List Available Datasets
Display the available datasets and their descriptions.

In [ ]:
for i, dataset in enumerate(AVAILABLE_DATASETS):
    print(f"{i+1}. {dataset['name']}: {dataset['description']}")

1. multiples_stocks: **Capital IQ Financials**

Structured financial data from S&P Capital IQ Pro.
2. transcripts_filings_agent: **Transcripts & Filings**

Earnings call transcripts & filings.
3. financials_market_cap: **Financials, Market Data, Segments, Relationships**
4. corporate_transactions: **Corporate Transactions SQL**

M&A, buybacks, placements, bankruptcies.
5. news_and_key_developments: **News & Key Developments SQL**

Business news & announcements.
6. professionals: **Executive Profiles SQL**

Board & executive profiles, roles, comp.


## Select Datasets and Enter Query
You can select datasets by their index (comma-separated) or leave empty to search all. Enter your search query below.

In [ ]:
# User input for dataset selection and query
selected_indices = input("Enter dataset numbers to search (comma-separated, or leave empty for all): ")
if selected_indices.strip():
    indices = [int(idx.strip())-1 for idx in selected_indices.split(",") if idx.strip().isdigit()]
    selected_datasets = [AVAILABLE_DATASETS[i]["name"] for i in indices if 0 <= i < len(AVAILABLE_DATASETS)]
else:
    selected_datasets = []  # All datasets

# i.e. "Quarterly EBITDA from Q3 of 2022 to Q1 of 2024 for Alibaba"
query = input("Enter your search query (default: 'Quarterly EBITDA from Q3 of 2022 to Q1 of 2024 for Alibaba'): ")
if not query.strip():
    query = "Quarterly EBITDA from Q3 of 2022 to Q1 of 2024 for Alibaba"

Enter dataset numbers to search (comma-separated, or leave empty for all): 
Enter your search query (default: 'Quarterly EBITDA from Q3 of 2022 to Q1 of 2024 for Alibaba'): 


## Run Search
This will call the Adaptive Search Service API and display the results as DataFrames.

In [ ]:
from typing import List, Dict

async def search_adaptive_search_service(query: str, selected_datasets) -> dict:
    """Call the Adaptive Search Service API with the provided query and selected datasets asynchronously using aiohttp.

    Args:
        query (str): The search query to send to the Adaptive Search Service API.
        selected_datasets (list): List of datasets to restrict the search to.

    Returns:
        tuple[list[GroundedData], list]: A tuple containing:
            - The parsed GroundedData models from the API response
            - The raw JSON response from the Adaptive Search Service API
    """
    access_token = get_adaptive_search_access_token()
    payload = (
        {"query": query, "allowed_datasets": selected_datasets}
        if selected_datasets
        else {"query": query}
    )

    max_retries = 3
    retry_delay = 2  # Initial delay in seconds

    for attempt in range(max_retries + 1):
        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(ADAPTIVE_SEARCH_API_HOST + "/api/v2/search",
                                        json=payload,
                                        headers={
                                            "Content-Type": "application/json",
                                            "Authorization": f"Bearer {access_token}"
                                        },
                                        timeout=60) as response:
                    response.raise_for_status()
                    adaptive_search_results = await response.json()
                    if not adaptive_search_results.get("response"):
                        return []
                    return adaptive_search_results["response"]
        except (requests.exceptions.RequestException, ValueError, aiohttp.ClientResponseError) as e:
            if getattr(e, "status", None) == 401:
                access_token = get_adaptive_search_access_token()
            if attempt == max_retries - 1:  # Last attempt
                print(f"Failed after {max_retries} attempts: {str(e)}")
                return "Error: " + str(e), []
            await asyncio.sleep(retry_delay * (attempt + 1))  # Exponential backoff

def extract_adaptive_search_result_data(res: Dict) -> tuple[pd.DataFrame, List[Dict]]:
    """Extract data from an Adaptive Search service result into a pandas DataFrame and list of Source objects.

    Args:
        res (Dict): A result from the Adaptive Search service

    Returns:
        tuple: (DataFrame containing the result data, list of dicts containing metadata like name, description, link)
    """
    if res.get("data"):

        df = pd.DataFrame(res["data"])

        sources: List[Dict] = []
        for source in res["sources"]:
            if source.get("source") and source["source"].get("uri"):
                source_data = {
                    "name": source["source"]["name"],
                    "link": source["source"]["uri"],
                    "description": f"Provider: {source['source']['provider']}",
                    "notes": [],
                    "limitations": [],
                }
                sources.append(source_data)

        return df, sources
    return None, []

In [ ]:
if query.strip():
    print("Searching... (this may take a few seconds)")
    start_time = time.time()
    # Use await for coroutine in Jupyter
    results = await search_adaptive_search_service(query, selected_datasets)
    elapsed_time = time.time() - start_time
    print(f"API call took {elapsed_time:.2f} seconds.")
    if not results or not isinstance(results, list) or len(results) == 0:
        print("No results found for your query.")
    else:
        print("Search completed successfully!")
        for i, res in enumerate(results):
            print(f"\nResult {i+1}")
            df, sources = extract_adaptive_search_result_data(res)
            display(df)
            for j, source in enumerate(sources):
                print(f"Source {j+1}:")
                print(f"  source: {source['name']}")
                print(f"  description: {source['description']}")
                print(f"  link: {source['link']}")
else:
    print("No query entered.")

Searching... (this may take a few seconds)
API call took 16.50 seconds.
Search completed successfully!

Result 1


,Company Name,Line Item,Period,Value
0,BABA,Ebitda,2022Q3,35013000000.000000
1,BABA,Ebitda,2022Q4,53200000000.000000
2,BABA,Ebitda,2023Q1,34156000000.000000
3,BABA,Ebitda,2023Q2,53681000000.000000
4,BABA,Ebitda,2023Q3,42407000000.000000
5,BABA,Ebitda,2023Q4,52331000000.000000
6,BABA,Ebitda,2024Q1,32674000000.000000


Source 1:
  source: BABA Ebitda
  description: Provider: 
  link: https://kfinance.kensho.com/api/v1/line_item/42083601/ebitda/quarterly/2022/2024/3/1


## View Raw API Response
You can inspect the full JSON response below if needed.

In [ ]:
import json
print(json.dumps(results, indent=2))

[
  {
    "schema": {
      "fields": [
        {
          "name": "Company Name",
          "type": "string"
        },
        {
          "name": "Line Item",
          "type": "string"
        },
        {
          "name": "Period",
          "type": "string"
        },
        {
          "name": "Value",
          "type": "string"
        }
      ]
    },
    "data": [
      {
        "Company Name": "BABA",
        "Line Item": "Ebitda",
        "Period": "2022Q3",
        "Value": "35013000000.000000"
      },
      {
        "Company Name": "BABA",
        "Line Item": "Ebitda",
        "Period": "2022Q4",
        "Value": "53200000000.000000"
      },
      {
        "Company Name": "BABA",
        "Line Item": "Ebitda",
        "Period": "2023Q1",
        "Value": "34156000000.000000"
      },
      {
        "Company Name": "BABA",
        "Line Item": "Ebitda",
        "Period": "2023Q2",
        "Value": "53681000000.000000"
      },
      {
        "Company Name": "BAB